<a href="https://colab.research.google.com/github/SilviaPorciuleanu/west-narratives-party-manifestos/blob/main/O3_Actor_identification_and_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sentence Validation and identification of references to EU from Europe


In [ ]:
# pip install openai pandas tqdm

import os
import json
import time
import io
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from google.colab import files
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))



Files

In [ ]:
from google.colab import drive
import hashlib
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Upload file
# Project folder on Drive
PROJECT_DIR = "/content/drive/MyDrive/West_Narratives"

# Input/output paths
INPUT_FILE = f"{PROJECT_DIR}/M_CE_west_sample_dictionaryraw_07052026.csv"
OUTPUT_FILE = f"{PROJECT_DIR}/M_CE_west_sample_dictionaryraw_07052026_chatgpt.csv"

TEXT_COLUMN = "keyword"
ID_COLUMN = "docname"

df = pd.read_csv(INPUT_FILE)



In [ ]:
MODEL = "gpt-5.4-mini-2026-03-17"

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "referent_category": {
            "type": "string",
            "enum": [
                "West",
                "Western countries",
                "Western signifier",
                "EU, NATO and European structures",
                "none"
            ]
        }
    },
    "required": ["referent_category"],
    "additionalProperties": False
}

In [ ]:
def build_prompt(sentence):
    return f"""
You are coding sentences from party manifestos in Central and Eastern Europe.

Task:
Identify what type of West-related referent appears in the sentence.

Return only valid JSON with:
- referent_category: "West", "Western countries", "Western signifier", "EU, NATO and European structures", or "none"

Referent categories:

"West" = explicit general Western category:
the West, Western Europe, Western world, Western democracies, Western civilization, Western countries, modern Western countries, etc.

Important:
- The phrase "Western countries" belongs to the "West" category, not to the "Western countries" category.
- If multiple references together suggest the broader Global West or Western geopolitical/civilizational space, code "West".

"Western countries" = specific mentions of one or more Western countries such as United States, Canada, Austria, Belgium, Cyprus, Denmark, Finland, France, Germany, Greece, Ireland, Italy, Luxembourg, Malta, Netherlands, Portugal, Spain, Sweden, United Kingdom, Switzerland etc

"Western signifier" = broad cultural, ideological, or geopolitical references to Europe, such as pro-European, European values, European identity, European-style, transatlantic space, or Euro-Atlantic orientation, excluding direct references to EU institutions, NATO, EU regulations, funds, or administrative structures.

"EU, NATO and European structures" = direct references to the European Union, EU institutions, European institutions, European Court of Human Rights, European Central Bank, NATO, EU directives, EU programmes, EU funds, European Structural Funds, or other formal European/Euro-Atlantic institutional frameworks.
"none" = no meaningful West-related referent is present.

Important coding rules:
- Do NOT code "Western Balkans" itself as "West". However, if the same sentence also mentions Europe, European integration, the EU, NATO, or another West-related referent, code that other referent normally.
- Do NOT code "Western Bulgaria", "western region", "western part of the country", or similar purely geographic uses as "West"; code them as "none" unless another West-related referent is present.
- Code EU funds, European Structural Funds, EU programmes, EU directives, EU regulations, EU administrative mechanisms, or sectoral EU policy instruments, European conventions, European Court of Human Rights, European Central Bank and other common European institutional frameworks and European conventions etc as "EU, NATO and European structures".
- Code minorities from Western countries such as German minority as "none".
- Code Central and Eastern Europe references as "none" unless another West or Europe-related referent is present.
- Code generic mentions of Europe, European countries etc as "Western signifier".
- Code generic lists of countries as "Western countries" only when they include specific named Western countries used as countries or political actors.
- Do NOT code treaties, conventions, or agreements as "Western countries" just because their names contain Western city names, such as Paris, Washington, Copenhagen, Maastricht, Lisbon, Geneva, Hague, or Schengen.
- Code generic lists of countries as "Western countries" only when they include specific named Western countries.
- If there is no West, named Western country, or Western signifier, code "none".
- take into account also when Europe or West are used as adjectives: pro-European or Western-style;

Sentence:
\"\"\"{sentence}\"\"\"

Return only valid JSON.
"""

In [ ]:
def code_sentence(sentence, max_retries=3):
    if pd.isna(sentence) or str(sentence).strip() == "":
        return {
            "referent_category": None
        }

    prompt = build_prompt(str(sentence))

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL,
                temperature=0,
                input=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "west_suitability",
                        "schema": schema
                    }
                }
            )

            return json.loads(response.output_text)

        except Exception as e:
            if attempt == max_retries - 1:
                return {
                    "referent_category": None,
                    "error": str(e)
                }
            time.sleep(2 ** attempt)

In [ ]:
df = pd.read_csv(INPUT_FILE)

df["referent_category"] = None
df["api_error"] = None

In [ ]:
for i in tqdm(range(len(df))):
    result = code_sentence(df.loc[i, TEXT_COLUMN])

    if result.get("error") is not None:
        print(f"\nAPI error at row {i}")
        print("Sentence:")
        print(df.loc[i, TEXT_COLUMN])
        print("\nError:")
        print(result.get("error"))

        df.to_csv(OUTPUT_FILE, index=False)
        break
    df.loc[i, "referent_category"] = result.get("referent_category")
    df.loc[i, "api_error"] = result.get("error")

    if i % 100 == 0:
        df.to_csv(OUTPUT_FILE, index=False)

else:
    df.to_csv(OUTPUT_FILE, index=False)
    print("Finished without API errors.")

100%|██████████| 100/100 [01:44<00:00,  1.04s/it]

Finished without API errors.


In [ ]:
df.head(100)

,docname,from,to,pre,keyword,post,pattern,lenght,row_id,actor,referent_category,api_error
0,88621_201210,2317,2317,"Until now, three institutions - the Navy (MoD)...","Now, following the Finnish example, the infras...","This means no more duplication of resources, m...",Finland,28,1,2,Western countries,None
1,88820_201610,828,828,We will help municipalities organise food wast...,"Following France's example, we will introduce ...",2.,France,31,2,2,Western countries,None
2,88621_200810,2357,2357,It would cost little or nothing to keep them t...,"Teachers could travel with their families, imp...",Such pedagogical work could also be integrated...,UK,19,3,0,none,None
3,88820_201610,2046,2046,Strengthening such a format would both protect...,The core objectives of the bilateral relations...,We will mobilise support for the EU's eastern ...,Europe,61,4,3,Western signifier,None
4,88621_201210,276,276,We hope that the situation in the euro area cl...,The European Union has been through several cr...,The recent decisions taken by the European Uni...,Europe,25,5,4,"EU, NATO and European structures",None
...,...,...,...,...,...,...,...,...,...,...,...,...
95,97330_200809,920,920,In addition to the implementation of the elect...,"4. within the European Union, we will advocate...",5. we will actively participate in the optimis...,Europe,27,96,4,"EU, NATO and European structures",None
96,92435_201110,427,427,"In the European Parliament, which is slowly be...","Thanks to the actions of our government, in th...",We have significantly improved relations with ...,Europe,66,97,4,"EU, NATO and European structures",None
97,86421_201804,150,150,Good morning!,"It's Good Friday, so it's worth thinking a lit...","A Christian Europe, which is Christian for our...",Europe,19,98,3,Western signifier,None
98,92720_201510,389,389,"In this regard, we will listen carefully to Po...",Poland should actively build coalitions to der...,Poland should resist the inflation of laws in ...,Europe,9,99,4,"EU, NATO and European structures",None


In [ ]:
recode_map = {
    "none": 0,
    "West": 1,
    "Western countries": 2,
    "Western signifier": 3,
    "EU, NATO and European structures": 4
}

df["referent_code"] = df["referent_category"].map(recode_map)

First round evaluation metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

In [ ]:
# Make sure both columns are numeric
df["referent_code"] = pd.to_numeric(df["referent_code"], errors="coerce")
df["actor"] = pd.to_numeric(df["actor"], errors="coerce")

# Keep only rows where both manual and predicted labels exist
eval_df = df.dropna(subset=["referent_code", "actor"]).copy()

# True labels = manual coding
y_true = eval_df["actor"].astype(int)

# Predicted labels = ChatGPT/API coding
y_pred = eval_df["referent_code"].astype(int)

In [ ]:
labels = [0, 1, 2, 3, 4]

label_names = {
    0: "none",
    1: "West",
    2: "Western countries",
    3: "Western signifier",
    4: "EU, NATO and European structures"
}

target_names = [label_names[i] for i in labels]

In [ ]:
# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
balanced_accuracy = balanced_accuracy_score(y_true, y_pred)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average="macro", zero_division=0
)

precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average="micro", zero_division=0
)

precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average="weighted", zero_division=0
)

metrics_summary = pd.DataFrame({
    "metric": [
        "accuracy",
        "balanced_accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_weighted",
        "recall_weighted",
        "f1_weighted"
    ],
    "value": [
        accuracy,
        balanced_accuracy,
        precision_macro,
        recall_macro,
        f1_macro,
        precision_micro,
        recall_micro,
        f1_micro,
        precision_weighted,
        recall_weighted,
        f1_weighted
    ]
})

metrics_summary

,metric,value
0,accuracy,0.940000
1,balanced_accuracy,0.934283
2,precision_macro,0.901595
3,recall_macro,0.934283
4,f1_macro,0.916294
5,precision_micro,0.940000
6,recall_micro,0.940000
7,f1_micro,0.940000
8,precision_weighted,0.944011
9,recall_weighted,0.940000


Category metrics

In [ ]:
report_df = pd.DataFrame(
    classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )
).transpose()

report_df

,precision,recall,f1-score,support
none,0.666667,0.800000,0.727273,5.00
West,1.000000,1.000000,1.000000,7.00
Western countries,0.947368,1.000000,0.972973,18.00
Western signifier,0.916667,0.956522,0.936170,23.00
"EU, NATO and European structures",0.977273,0.914894,0.945055,47.00
accuracy,0.940000,0.940000,0.940000,0.94
macro avg,0.901595,0.934283,0.916294,100.00
weighted avg,0.944011,0.940000,0.940994,100.00


In [ ]:
# View mistakes by category
eval_df["y_true"] = y_true.values
eval_df["y_pred"] = y_pred.values

mistakes = eval_df[eval_df["y_true"] != eval_df["y_pred"]].copy()

mistakes[[
    TEXT_COLUMN,
    "actor",
    "referent_code",
    "referent_category"
]].head(100)

,keyword,actor,referent_code,referent_category
9,"The continuation of decentralisation, a proper...",0,2,Western countries
11,The Greens want to prepare and enforce this pr...,4,3,Western signifier
57,Efforts to ensure that the two strategic Europ...,4,0,none
68,Bulgaria will contribute to the acceleration o...,4,0,none
78,"At the same time, the road will become an impo...",3,4,"EU, NATO and European structures"
99,We will promote our country's representatives ...,4,3,Western signifier


In [ ]:
OUTPUT_FILE = "manifesos_west_chatgpt_annotate_sample_actor.csv"

df.to_csv(OUTPUT_FILE, index=False)

from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Coding the full dataset

In [ ]:
def run_api_coding(input_file, output_file, text_column, model=MODEL):
    df = pd.read_csv(input_file)

    df["referent_category"] = None
    df["api_error"] = None

    for i in tqdm(range(len(df))):
        result = code_sentence(df.loc[i, text_column])

        if result.get("error") is not None:
            print(f"\nAPI error at row {i}")
            print("Sentence:")
            print(df.loc[i, text_column])
            print("\nError:")
            print(result.get("error"))

            df.to_csv(output_file, index=False)
            break

        df.loc[i, "referent_category"] = result.get("referent_category")
        df.loc[i, "api_error"] = result.get("error")

        if i % 100 == 0:
            df.to_csv(output_file, index=False)

    else:
        df.to_csv(output_file, index=False)
        print("Finished without API errors.")

    return df

In [ ]:
INPUT_FILE = f"{PROJECT_DIR}/manifesos_west_dictionary_based_unique_07052026.csv"
OUTPUT_FILE = f"{PROJECT_DIR}/manifesos_west_dictionary_based_unique_07052026_chatgpt.csv"
TEXT_COLUMN = "keyword"

df_full_coded = run_api_coding(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
    text_column=TEXT_COLUMN,
    model="gpt-5.4-mini-2026-03-17"
)

100%|██████████| 12513/12513 [3:48:24<00:00,  1.10s/it]


Finished without API errors.


In [ ]:
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.to_csv(OUTPUT_FILE, index=False)

from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>